# Market Maker Implementation

## Logarithmic Market Scoring Rule (LMSR)

We will implement the LMSR according to it's cost-function formulation.

The cost function, for a market quantity $q$ and liquidity parameter $b$ is:

$$C(q) = b \; \texttt{log}(\sum_j e^{q_j/b})$$

The price of purchasing a bundle $r$ is:

$$\texttt{cost}(r|q) = C(q+r) - C(q)$$

Finally, the instantaneous price of security $i$ is:

$$p_i = \frac{\partial C}{\partial q_i} = \frac{e^{q_i/b}}{\sum_j e^{q_j/b}}$$

Reference: [Y. Chen and D. M. Pennock, "A Utility Framework for Bounded-Loss Market Makers," in Proceedings of the 23rd Conference on Uncertainty in Artificial Intelligence (UAI), 2007, pp. 49-56.](https://arxiv.org/abs/1206.5252)

In [37]:
import numpy as np

class LMSR:
    q = None
    b = None
    money = None

    def __init__(self, k: int = 10, b: float = 1.0, starting_money: float = 0.0):
        self.q = np.zeros(k)
        self.b = b
        self.money = starting_money

    def __repr__(self):
        return f"LMSR(q={self.q}, b={self.b}, money={self.money})"

    def cost(self, q: np.ndarray = None):
        if q is None:
            q = self.q
        return self.b * np.log(np.sum(np.exp(q/self.b)))

    def prices(self):
        total = np.sum(np.exp(self.q/self.b))
        return np.exp(self.q/self.b)/total

    def price_bundle(self, r: np.ndarray):
        return self.cost(self.q+r) - self.cost(self.q)

    def purchase_bundle(self, r: np.ndarray):
        # Never hold negative quantities
        if np.min(self.q + r) < 0:
            raise ValueError('Unsufficient market quantity to purchase bundle.')

        price = self.price_bundle(r)
        self.q += r
        self.money += price
        return price

In [38]:
lmsr = LMSR()
print(lmsr)

bundle = np.ones(10) * 1

for _ in range(10):
    print(f'Trader pays: ${lmsr.purchase_bundle(bundle):.2f} for r={bundle}')
    print(lmsr)

bundle *= -1
for _ in range(10):
    print(f'Trader pays: ${lmsr.purchase_bundle(bundle):.2f} for r={bundle}')
    print(lmsr)

LMSR(q=[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.], b=1.0, money=0.0)
Trader pays: $1.00 for r=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
LMSR(q=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.], b=1.0, money=0.9999999999999996)
Trader pays: $1.00 for r=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
LMSR(q=[2. 2. 2. 2. 2. 2. 2. 2. 2. 2.], b=1.0, money=2.0)
Trader pays: $1.00 for r=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
LMSR(q=[3. 3. 3. 3. 3. 3. 3. 3. 3. 3.], b=1.0, money=3.0)
Trader pays: $1.00 for r=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
LMSR(q=[4. 4. 4. 4. 4. 4. 4. 4. 4. 4.], b=1.0, money=4.0)
Trader pays: $1.00 for r=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
LMSR(q=[5. 5. 5. 5. 5. 5. 5. 5. 5. 5.], b=1.0, money=5.0)
Trader pays: $1.00 for r=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
LMSR(q=[6. 6. 6. 6. 6. 6. 6. 6. 6. 6.], b=1.0, money=5.999999999999999)
Trader pays: $1.00 for r=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
LMSR(q=[7. 7. 7. 7. 7. 7. 7. 7. 7. 7.], b=1.0, money=6.999999999999999)
Trader pays: $1.00 for r=[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
LMSR(q=[8. 8. 8. 8. 8. 8. 8. 8. 8. 8.